# NACC data importation, analysis and preparation

Created by: Sahana Kowshik

## Libraries and Data Files
- Import necessary libraries, `pandas` and `numpy`.
- Load several CSV files containing data and metadata for analysis.

In [1]:
import pandas as pd
pd.set_option('display.max_columns', None)
import numpy as np
import warnings
warnings.filterwarnings("ignore")
import json
import string
import ast
from tqdm import tqdm
from math import floor, ceil

In [2]:
directory_path = "/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nacc"

## Diactionary Cleaning

- Fill missing values in variable_id column using forward filling method.
- Merge `nacc_variable` with `nacc_allowable_code` on variable IDs to consolidate variable descriptions and their corresponding codes.
- Filter merged data for specific UDS forms ('A1', 'A3', 'A4', 'A5', 'B1', 'B1a', 'B2','B3', 'B5', 'B6', 'B7', 'B8', 'C1', 'C1, C2', 'C1, C2, C2T', 'C1,C2', 'C2, C2T', 'C2', 'C2T', 'G1', 'D2', 'I', 'CSF') to get clinical features.
- Filter merged data for specific UDS forms (B9 and D1) to get diagnosis features.

In [3]:
form = {
    'U1': "",
    "U2": "",
    "T1": "",
    'A1': 'Subject Demographics',
    'A2': 'Co-participant Demographics',
    'A3': 'Subject Family History',
    'A4': 'Subject Medications',
    'A5': 'Subject Health History',
    'B1': 'Physical',
    'B2': 'His and CVD',
    'B3': "Unified Parkinson's Disease Rating Scale (UPDRS)",
    'B4': 'Clinical Dementia Rating (CDR)',
    'B5': 'Neuropsychiatric Inventory Questionnaire (NPI-Q)',
    'B6': 'Geriatric Depression Scale (GDS)',
    'B7': 'Functional Assessment Scale (FAS)',
    'B8': 'Physical/Neurological Exam Findings',
    'B9': 'Clinician Judgment of Symptoms',
    'C': 'Neuropsychological Battery Summary Scores',
    'D1': 'Clinician Diagnosis',
    'D2': 'Clinician-assessed Medical Conditions',
    'G1': 'Genetic testing',
    'I': 'Imaging evidence',
    'CSF': 'CSF evidence',
    "Z1X": "",
    "CLS": "",
    "U3": "",
}

inverse_form = {
    v: k for k, v in form.items()
}

In [4]:
with open(f"{directory_path}/NACC_dictionary.json", 'r') as json_file:
    merged_dict = json.load(json_file)

In [50]:
patient_forms = ['A1', 'A3', 'A4', 'A5', 'B1', 'B1a', 'B2', 'B3', 'B5', 'B6', 'B7', 'B8', 'C', 'G1', 'D2', 'I', 'CSF']
patient_dict = {
    key: value for key, value in merged_dict.items() if value['FORM ID'] in patient_forms
}

diag_forms = ['B9', 'D1']
diag_dict = {
    key: value for key, value in merged_dict.items() if value['FORM ID'] in diag_forms
}

In [40]:
patient_dict['BIRTHMO']

{'FORM ID': 'A1',
 'Description': "Subject's month of birth",
 'Values': {'range': [1.0, 12.0]}}

## Load data

In [9]:
all_nacc = pd.read_csv('/projectnb/vkolagrp/datasets/NACC/csv/processed/investigator_ftldlbd_merged_vnum_unique_nacc65.csv')

In [10]:
def get_latest_visits(data):
    result = data.sort_values(by=['NACCID', 'NACCVNUM'], ascending=[True, False])
    result = result.drop_duplicates(subset='NACCID', keep='first').reset_index(drop=True)
    return result

all_nacc_latest = get_latest_visits(all_nacc)
all_nacc_latest['COHORT'] = "NACC"

In [15]:
replacements = {
    '.F': np.nan, '0': np.nan, '0.0': np.nan, 
    '8': np.nan, '8.0': np.nan, 'SPECT': np.nan, 
    'spect': np.nan, 'CT': np.nan, '3/3': np.nan,
    'HMPAO-SPECT': np.nan, 'SPECT-HMPAO': np.nan, 
    'BRAIN SPECT': np.nan, 'brain spect': np.nan, 
    'CT BRAIN': np.nan, 'sMRI; outside SPECT scan': np.nan, 
    'Technetium Neurolite Perfusion': np.nan, '(Spect) 2004': np.nan, 
    'SPECT 2017 NON SPECIFIC': np.nan, 'Proton spectroscopy MR': np.nan, 
    'SMRI': np.nan, 'Spect': np.nan, 'SPECT-ECD': np.nan, 
    'Spect-ECD': np.nan, 'HMPAO SPECT': np.nan, 'EDC SPECT': np.nan, 
    'ECD SPECT': np.nan, 'HMPAD-SPECT:HYPERFUSION': np.nan, 'Technetium Neurolite Perfusion (Spect) 2004': np.nan, 'LT 2-25-2015': np.nan, 'APO E 3/3': np.nan, 'APO E 4/4': np.nan
}

# Replace the values in the DataFrame
all_nacc_latest = all_nacc_latest.replace({'-4': np.NaN, -4: np.NaN, '-4.0': np.NaN, -4.0: np.NaN})
all_nacc_latest['OTHBIOMX'] = all_nacc_latest['OTHBIOMX'].replace(replacements)
all_nacc_latest.loc[all_nacc_latest['OTHBIOMX'].str.contains('apoe', case=False, na=False), 'OTHBIOMX'] = np.nan

In [16]:
all_nacc_latest.drop(['PROBAD', 'PROBADIF', 'POSSAD', 'POSSADIF', 'FTLDSUBX', 'OTHBIOM', 'NACCC1', 'NACCC2', 'NEWINF','NACCAGEB', 'WHODIDDX', 'DXMETHOD'], axis=1, inplace=True)

## JSON Dictionary Creation

- Iterate through the data and create patient and diagnosis json files for each row.

In [51]:
# patient_code_dict.update({f"DRUG{i}" : patient_code_dict["DRUG1-DRUG40"] for i in range(1,41)})
patient_dict.update({f"DRUG{i}" : patient_dict["DRUG1-DRUG40"] for i in range(1,41)})
patient_dict = {k: v for k, v in patient_dict.items() if k in all_nacc_latest.columns}

In [52]:
patient_dict['DRUG2']

{'FORM ID': 'A4',
 'Description': 'Name of medications used within two weeks of UDS visit',
 'Values': {'-4': 'Not available: UDS form submitted did not collect data in this way, or a skip pattern precludes response to this question',
  '<UND>': 'Any text or numbers'}}

In [ ]:
def get_json(row, data_dict, column, unknown):
    """
    Convert a row of patient data into a JSON summary using a data dictionary.
    Skips unknown, not applicable, or not available values.
    Tracks unknown values and values outside of expected ranges.
    """
    patient_data = {}
    for k, v in dict(row).items():
        # Skip columns not in the data dictionary or the NACCID column
        if k not in data_dict or k == "NACCID":
            continue

        # Only process non-NaN values
        if not pd.isna(v):
            # If value is a string, check for various "unknown" or "not available" markers
            if isinstance(v, str):
                if (
                    ('unknown' in v.lower()) or
                    ('not applicable' in v.lower()) or
                    ('not available' in v.lower()) or
                    ('did not report' in v.lower()) or
                    ('not assessed' in v.lower()) or
                    ('none reported' in v.lower()) or
                    ('did not answer' in v.lower())
                ):
                    continue
            # If value is in the dictionary, check if its label is "unknown" etc.
            elif str(v) in data_dict[k]['Values']:
                if (
                    ('unknown' in data_dict[k]['Values'][str(v)].lower()) or
                    ('not applicable' in data_dict[k]['Values'][str(v)].lower()) or
                    ('not available' in data_dict[k]['Values'][str(v)].lower()) or
                    ('did not report' in data_dict[k]['Values'][str(v)].lower()) or
                    ('not assessed' in data_dict[k]['Values'][str(v)].lower()) or
                    ('none reported' in data_dict[k]['Values'][str(v)].lower()) or
                    ('did not answer' in data_dict[k]['Values'][str(v)].lower())
                ):
                    continue
            # If value is a number, check its integer representation in the dictionary
            elif str(int(v)) in data_dict[k]['Values']:
                if (
                    ('unknown' in data_dict[k]['Values'][str(int(v))].lower()) or
                    ('not applicable' in data_dict[k]['Values'][str(int(v))].lower()) or
                    ('not available' in data_dict[k]['Values'][str(int(v))].lower()) or
                    ('did not report' in data_dict[k]['Values'][str(int(v))].lower()) or
                    ('not assessed' in data_dict[k]['Values'][str(int(v))].lower()) or
                    ('none reported' in data_dict[k]['Values'][str(int(v))].lower()) or
                    ('did not answer' in data_dict[k]['Values'][str(int(v))].lower())
                ):
                    continue
            # If value is not in the dictionary and not a range, add to unknowns
            else:
                if "range" not in data_dict[k]['Values']:
                    unknown.append((row['NACCID'], k, v))
        
        # Get the form name for this variable
        form_name = form[data_dict[k]['FORM ID']]
        # Initialize the form section if not already present
        if form_name not in patient_data:
            patient_data[form_name] = {}
        
        # Build the key for this variable, including range if appropriate
        if (
            (k in data_dict) and
            ("range" in data_dict[k]['Values']) and
            ("Demographics" not in form_name) and
            ("year" not in data_dict[k]["Description"].lower())
        ):
            key = data_dict[k]["Description"] + " (range: " + str(data_dict[k]['Values']["range"]).replace(',', ' -') + ')'
        else:
            key = data_dict[k]["Description"]
        
        # Add the value to the patient_data dictionary
        if not pd.isna(v):
            if isinstance(v, str):
                # If the key already exists, append the new value
                if key in patient_data[form_name]:
                    patient_data[form_name][key] += f", {v}"
                else:
                    patient_data[form_name][key] = v
            # If value is in the dictionary, use its label
            elif str(v) in data_dict[k]['Values']:
                patient_data[form_name][key] = data_dict[k]['Values'][str(v)]
            # If integer value is in the dictionary, use its label
            elif str(int(v)) in data_dict[k]['Values']:
                patient_data[form_name][key] = data_dict[k]['Values'][str(int(v))]
            # If value is within the expected range, use the value itself
            elif (
                "range" in data_dict[k]['Values'] and
                data_dict[k]['Values']["range"][0] <= v <= data_dict[k]['Values']["range"][1]
            ):
                patient_data[form_name][key] = v
            # If value is not in the dictionary and not in range, track as unknown or out-of-range
            else:
                if "range" not in data_dict[k]['Values']:
                    unknown.append((row['NACCID'], k, v))
                else:
                    # Add to check list for further inspection
                    check.append((row['NACCID'], k, v, data_dict[k]['Values']["range"][0], data_dict[k]['Values']["range"][1]))
    
    # Remove empty form sections
    patient_data = {k: v for k, v in patient_data.items() if v}
    
    # Remove co-participant information if present
    if 'Co-participant Demographics' in patient_data:
        del patient_data['Co-participant Demographics']
    
    # Remove or reorder certain subject demographic fields
    if 'Subject Demographics' in patient_data:
        # Remove month of birth if present
        if "Subject's month of birth" in patient_data['Subject Demographics']:
            del patient_data['Subject Demographics']["Subject's month of birth"]
        # Remove year of birth if present
        if "Subject's year of birth" in patient_data['Subject Demographics']:
            del patient_data['Subject Demographics']["Subject's year of birth"]
        # Move age at visit to the top if present
        if "Subject's age at visit" in patient_data['Subject Demographics']:
            key_to_move_first = "Subject's age at visit"
            patient_data['Subject Demographics'] = {
                key_to_move_first: patient_data['Subject Demographics'][key_to_move_first],
                **{k: v for k, v in patient_data['Subject Demographics'].items() if k != key_to_move_first}
            }
    
    # Order the patient_data according to the form order
    key_order = [k for k in form.values() if k in patient_data]
    sorted_patient_data = {k: patient_data[k] for k in key_order if k in patient_data}
    
    # Store the JSON string in the specified column
    row[column] = json.dumps(sorted_patient_data)
    return row

In [31]:
patient_unknown = []
diag_unknown = []

In [32]:
check = []

In [45]:
from tqdm import tqdm
tqdm.pandas()

all_nacc_latest = all_nacc_latest.progress_apply(
    get_json,
    data_dict=patient_dict,
    column='patient_summary',
    unknown=patient_unknown,
    axis=1
)

100%|██████████| 50962/50962 [03:41<00:00, 229.62it/s]


In [ ]:
json.loads(x.iloc[0]['patient_summary'])

In [46]:
check

[('NACC108239', 'NACCBMI', 9.9, 10.0, 100.0),
 ('NACC524817', 'NACCBMI', 9.6, 10.0, 100.0),
 ('NACC615902', 'WEIGHT', 404.0, 50.0, 400.0),
 ('NACC926721', 'BPSYS', 69.0, 70.0, 230.0),
 ('NACC929474', 'BPSYS', 68.0, 70.0, 230.0),
 ('NACC954785', 'WEIGHT', 443.0, 50.0, 400.0)]

In [40]:
patient_unknown

[('NACC099600', 'SLEEPOTX', 0.0),
 ('NACC099600', 'SLEEPOTX', 0.0),
 ('NACC118701', 'UDSBENRS', 9.0),
 ('NACC118701', 'UDSBENRS', 9.0),
 ('NACC333633', 'ABUSX', 0.0),
 ('NACC333633', 'ABUSX', 0.0),
 ('NACC334084', 'OTHBIOMX', 0.0),
 ('NACC334084', 'OTHBIOMX', 0.0),
 ('NACC397335', 'MOCALANX', 0.0),
 ('NACC397335', 'MOCALANX', 0.0),
 ('NACC455382', 'UDSBENRS', 9.0),
 ('NACC455382', 'UDSBENRS', 9.0),
 ('NACC473141', 'NPSYLANX', -6.0),
 ('NACC473141', 'NPSYLANX', -6.0),
 ('NACC559449', 'NPSYLANX', 20.0),
 ('NACC559449', 'NPSYLANX', 20.0),
 ('NACC601725', 'TRACTLHX', 8.0),
 ('NACC601725', 'TRACTLHX', 8.0),
 ('NACC676475', 'NPSYLANX', 96.0),
 ('NACC676475', 'NPSYLANX', 96.0),
 ('NACC693493', 'PRIMLANX', 0.0),
 ('NACC693493', 'PRIMLANX', 0.0),
 ('NACC697004', 'NPSYLANX', -6.0),
 ('NACC697004', 'NPSYLANX', -6.0),
 ('NACC915172', 'ARTYPEX', 1.0),
 ('NACC915172', 'ARTYPEX', 1.0),
 ('NACC937260', 'RACEX', 0.0),
 ('NACC937260', 'RACEX', 0.0),
 ('NACC949359', 'ABUSX', 0.0),
 ('NACC949359', 'ABUSX'

In [53]:
from tqdm import tqdm
tqdm.pandas()

all_nacc_latest = all_nacc_latest.progress_apply(
    get_json,
    data_dict=diag_dict,
    column='diag_summary',
    unknown=diag_unknown,
    axis=1
)

100%|██████████| 50962/50962 [02:51<00:00, 297.71it/s]


In [42]:
diag_unknown

[('NACC090809', 'MOMODEX', 1.0),
 ('NACC090809', 'MOMODEX', 1.0),
 ('NACC215672', 'NACCCGFX', 0.0),
 ('NACC215672', 'NACCCGFX', 0.0),
 ('NACC321425', 'MOMODEX', 1.0),
 ('NACC321425', 'MOMODEX', 1.0),
 ('NACC432813', 'MOMODEX', 1.0),
 ('NACC432813', 'MOMODEX', 1.0),
 ('NACC481858', 'MOMODEX', 1.0),
 ('NACC481858', 'MOMODEX', 1.0),
 ('NACC538635', 'BEOTHRX', 0.0),
 ('NACC538635', 'BEOTHRX', 0.0),
 ('NACC546087', 'MOMODEX', 1.0),
 ('NACC546087', 'MOMODEX', 1.0),
 ('NACC650577', 'COGMODEX', 64.0),
 ('NACC650577', 'COGMODEX', 64.0),
 ('NACC674720', 'MOMODEX', 1.0),
 ('NACC674720', 'MOMODEX', 1.0),
 ('NACC689405', 'MOMODEX', 1.0),
 ('NACC689405', 'MOMODEX', 1.0),
 ('NACC836462', 'OTHMUTX', 0.0),
 ('NACC836462', 'OTHMUTX', 0.0)]

In [ ]:
json.loads(all_nacc_latest.iloc[0]['patient_summary'])

## Data Export

- Save the processed data.

In [44]:
len(all_nacc_latest)

50962

In [45]:
len(all_nacc_latest[all_nacc_latest['NACCUDSD'] != 2]['NACCID'].unique())

48876

In [46]:
import os
if not os.path.exists(f"{directory_path}"):
    os.makedirs(f"{directory_path}", exist_ok=True)

all_nacc_latest.to_csv(f"{directory_path}/NACC_wjson.csv", index=False)

In [47]:
def remove_empty_keys(text):
    # text = text.replace("Neuropsychological battery Summary Scores", "Neuropsychological Battery Summary Scores")
    patient_file_json = json.loads(text)
    # patient_file_json = {k: v for k, v in patient_file_json.items() if v}
    for k, v in patient_file_json.items():
        if not v:
            print(patient_file_json)
            print("found")
            raise
    return json.dumps(patient_file_json)

all_nacc_latest["patient_summary"] = all_nacc_latest["patient_summary"].apply(remove_empty_keys)